# Notebook 03 — DistilBERT Pretrained Model
**Model:** distilbert-base-uncased
**Approach:** Fine-tune on MCQ data — score each (prompt, option) pair
**Metric:** MAP@3

## Step 1 — Install

In [1]:
!pip install transformers datasets wandb --quiet

## Step 2 — Imports

In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
import wandb

OPTIONS = ['A', 'B', 'C', 'D', 'E']
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Step 3 — WandB Login

In [3]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('GENAI_T2_2026'))

wandb.init(
    project = '23f2003665-t22026',
    name= 'distilbert-v3',
    config  = {
        'model': 'distilbert-base-uncased',
        'max_len': 160,
        'batch_size' : 16,
        'epochs' : 10,
        'lr': 2e-5,
    }
)
cfg = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Step 4 — Load Data

In [4]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print('Train:', train_df.shape, '| Test:', test_df.shape)

Train: (2000, 8) | Test: (500, 7)


In [5]:
import re

# These phrases appear in many prompts and confuse the model
NOISE = [
    'pick the best possible answer',
    'select the most accurate option', 
    'identify the correct statement',
    'choose the correct answer',
    'determine the correct option',
    'which of the following is correct',
    'among the listed options',
    'from the following choices',
    'based on the given context',
    'carefully',
]

def clean_prompt(text):
    text = str(text).lower()
    for phrase in NOISE:
        text = text.replace(phrase, '')
    # Remove extra spaces and punctuation left behind
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to prompts
train_df['prompt'] = train_df['prompt'].apply(clean_prompt)
test_df['prompt'] = test_df['prompt'].apply(clean_prompt)

print('Noise removed from prompts.')
print('Example:', train_df['prompt'].iloc[0][:100])

Noise removed from prompts.
Example: what is martin heideggers view on the relationship between time and human existence


## Step 5 — Split and Expand
Split by question ID to avoid data leakage.
Expand each question into 5 (prompt, option) pairs.

In [6]:
# Split by question ID — not row index
train_ids, val_ids = train_test_split(
    train_df['id'].unique(), test_size=0.1, random_state=42
)
train_data = train_df[train_df['id'].isin(train_ids)].reset_index(drop=True)
val_data = train_df[train_df['id'].isin(val_ids)].reset_index(drop=True)

# Expand: 1 question → 5 (prompt, option, label) rows
def expand(df, is_test=False):
    rows = []
    for _, row in df.iterrows():
        for opt in OPTIONS:
            r = {'text1': str(row['prompt']),
                 'text2': str(row[opt])}
            if not is_test:
                r['label'] = 1.0 if row['answer'] == opt else 0.0
            rows.append(r)
    return rows

train_rows = expand(train_data)
val_rows = expand(val_data)
test_rows = expand(test_df, is_test=True)
print('Train pairs:', len(train_rows), '| Val pairs:', len(val_rows))

Train pairs: 9000 | Val pairs: 1000


## Step 6 — Tokenize
Convert to HuggingFace Dataset and tokenize — same as instructor.

In [7]:
MODEL_NAME = cfg.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenize (prompt, option) pairs
def tokenize(batch):
    return tokenizer(
        batch['text1'], batch['text2'],
        max_length=cfg.max_len, padding='max_length', truncation=True
    )

# Convert lists to HuggingFace datasets and tokenize
train_dataset = Dataset.from_list(train_rows).map(tokenize, batched=True)
val_dataset = Dataset.from_list(val_rows).map(tokenize, batched=True)
test_dataset = Dataset.from_list(test_rows).map(tokenize, batched=True)

# Set PyTorch format
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch',columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch',columns=['input_ids', 'attention_mask'])
print('Tokenization done.')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Tokenization done.


## Step 7 — Load Model

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type='regression'
).to(device)
print('Model loaded:', MODEL_NAME)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased


## Step 8 — MAP@3 Metric

In [9]:
def compute_metrics(eval_pred):
    logits, _ = eval_pred
    scores = torch.sigmoid(torch.tensor(logits)).numpy().reshape(-1, 5)
    aps = []
    for i in range(len(val_data)):
        correct = OPTIONS.index(val_data.iloc[i]['answer'])
        top3 = np.argsort(-scores[i])[:3]
        ap, hits = 0.0, 0
        for rank, idx in enumerate(top3):
            if idx == correct:
                hits += 1
                ap += hits / (rank + 1)
        aps.append(ap)
    return {'val_map3': float(np.mean(aps))}

## Step 9 — Train with Trainer
Trainer handles training loop, evaluation, and best checkpoint saving automatically.

In [10]:
args = TrainingArguments(
    output_dir = '/kaggle/working/',
    num_train_epochs = cfg.epochs,
    per_device_train_batch_size = cfg.batch_size,
    per_device_eval_batch_size  = cfg.batch_size,
    learning_rate = cfg.lr,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model = 'val_map3',
    greater_is_better = True,
    fp16= False,
    report_to= 'wandb',
)

trainer = Trainer(
    model= model,
    args= args,
    train_dataset= train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Val Map3
1,No log,0.168032,0.947500
2,0.226026,0.070653,0.977500
3,0.226026,0.036195,0.992500
4,0.067839,0.026044,0.987500
5,0.067839,0.013514,0.987500
6,0.029882,0.012213,0.995000
7,0.029882,0.008059,0.995000
8,0.012981,0.007311,0.995000
9,0.008905,0.006129,0.995000
10,0.008905,0.005810,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2820, training_loss=0.06208500507030081, metrics={'train_runtime': 806.7133, 'train_samples_per_second': 111.564, 'train_steps_per_second': 3.496, 'total_flos': 3725579145600000.0, 'train_loss': 0.06208500507030081, 'epoch': 10.0})

## Step 10 — Inference and Submission

In [11]:
# Trainer loads best checkpoint automatically
preds = trainer.predict(test_dataset)
test_scores = torch.sigmoid(torch.tensor(preds.predictions)).numpy().reshape(-1, 5)

rows = []
for i in range(len(test_df)):
    top3 = [OPTIONS[j] for j in np.argsort(-test_scores[i])[:3]]
    rows.append({'id': test_df.iloc[i]['id'], 'Prediction': ' '.join(top3)})

pd.DataFrame(rows).to_csv('/kaggle/working/submission_distilbert.csv', index=False)
print('Submission saved!')
pd.DataFrame(rows).head()

Submission saved!


,id,Prediction
0,1,A C D
1,2,B D C
2,3,B D A
3,4,E B C
4,5,C E D


## Step 11 — Finish WandB

In [12]:
wandb.finish()

eval/loss,█▄▂▂▁▁▁▁▁▁
eval/runtime,▁▄▇███████
eval/samples_per_second,█▅▂▁▁▁▁▁▁▁
eval/steps_per_second,█▅▂▁▁▁▁▁▁▁
eval/val_map3,▁▅▇▆▆▇▇▇▇█
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▂▂▃▃▃▄▄▅▆▆▆▇▇██
train/global_step,▁▂▂▃▃▃▄▄▅▆▆▆▇▇███
+3,...


In [13]:
# Save full model + tokenizer
trainer.model.save_pretrained('/kaggle/working/distilbert_final/')
tokenizer.save_pretrained('/kaggle/working/distilbert_final/')
print('DistilBERT saved.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT saved.
